In [ ]:
from fmsae_repro import MODELS, LEGACY_ANALYSIS, load_manifest

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['pdf.fonttype'] = 42

### Training FM-SAE with per-sample loss normalization (reproducing SAE1)

This notebook trains the SAE with settings closer to the Anthropic training notes (per-sample loss, fixed λ, inputs scaled to E[‖x‖²] = d) while reproducing the behavior of SAE1 trained in `train_SAE.ipynb`.

In [ ]:
import ezslide
import lazyslide as zs
import mesoslide as ms
import pandas as pd
import joblib 


First, we will load the slides, which now will contain the patch embeddings

In [ ]:
manifest = load_manifest()
slides = ezslide.open_slides(manifest, attach_images=False)

Each setting below is rescaled from the SAE1 configuration so that the optimization matches SAE1:

| Setting | SAE1 (`train_SAE.ipynb`) | This notebook | Why |
|---|---|---|---|
| `loss_normalization` | `legacy` (MSE over B·d, L1 summed over B) | `per_sample` (squared error summed over d, L1 summed over features; both averaged over B) | at equal λ ratio, the per-sample loss is d × the legacy loss |
| `input_norm` | `sqrt_d` (E[‖x‖²] = √d) | `d` (inputs × c, c = d^¼ ≈ 5.66) | |
| λ | adaptive, at floor 1e-6 after step 70 | fixed, 1e-6 · B · d · c · 1.13 ≈ 13.4 | B·d from loss normalization, c from input scale; 1.13 compensates for the controller raising λ to 1.34e-6 during the first 70 steps |
| `adam_eps` | 1e-8 | 1e-8 · d · c² | weight gradients are d·c² larger; Adam is scale-invariant except for eps |
| `bias_lr_scale`, `bias_adam_eps` | 1, 1e-8 | c, 1e-8 · d · c | biases must grow by c under input scaling, and their gradients scale by d·c |
| `grad_clip_norm` | 1.0 (never active: max grad norm 0.096) | None | clipping at 1.0 would be active on every step here and changes training substantially |

The per-sample, input-scaling, eps, and bias settings reproduce SAE1 exactly (bit-identical weights with the adaptive controller). The fixed λ is the only approximation.

In [ ]:
from mesoslide.tools.sparse_coding._sae import SparseAutoencoder

B, d = 2048, 1024
c = d ** 0.25  # input scale ratio between input_norm='d' and 'sqrt_d'

sae = SparseAutoencoder(
    expansion_factor=64,
    batch_size=B,
    num_steps=2000,
    input_norm='d',
    loss_normalization='per_sample',
    lambda_mode='fixed',
    l1_coefficient=1e-6 * B * d * c * 1.13,
    grad_clip_norm=None,
    adam_eps=1e-8 * d * c**2,
    bias_lr_scale=c,
    bias_adam_eps=1e-8 * d * c,
    learning_rate=5e-5,
    random_state=0,
)
sae.fit(slides, obsm_key="UNI_embedding", tile_key="tiles",
        device='cuda:1', verbose=100)
joblib.dump(sae, MODELS / "SAE" / "sae_model_per_sample_repro.joblib")


Now we apply the trained model to all slides to get feature scores for all SAE features

In [ ]:
for _, slide in slides.items():
    ms.tl.feature_extraction(
        slide,
        model="uni",
        tile_key="tiles",
        key_added="UNI_embedding",
        sparse=True,
        sparse_transform=sae.transform,
        sparse_key_added="UNI_SAE_repro",
        save=False,
    )

In [ ]:
from mesoslide.tools import FeatureSelector

selector = FeatureSelector(
    pct_threshold=0.001,
    max_score_threshold=0.01,
    n_chunks=40
)
selector.compute_activation_stats(slides, 
                                  feature_prefix='UNI_SAE_repro', 
                                  num_features=65536)
g = selector.plot_feature_selection()


In [ ]:
highly_active_features_idx = selector.get_selected_indices()
len(highly_active_features_idx)

In [ ]:
# Cluster
from mesoslide.tools import FeatureClusterer

clusterer = FeatureClusterer(high_activation_threshold=0.5)
clusterer.compute_iou(slides, feature_prefix='UNI_SAE_repro',
                      feature_indices=highly_active_features_idx)
clusterer.cluster(threshold=25, criterion='maxclust')

In [ ]:
g = clusterer.plot_heatmap()


In [ ]:
_ = clusterer.select_exemplars(slides)

In [ ]:
import distinctipy
from matplotlib.colors import ListedColormap

cmap = ListedColormap(distinctipy.get_colors(25, rng=43))

fig, axs = clusterer.plot_feature_gallery(
    cmap=cmap,
    fontsize=10,
    return_fig=True,
)

### Comparison with SAE1

Apply SAE1 (`sae_model_demo.joblib` from `train_SAE.ipynb`) to the same slides and compare the selected features.

In [ ]:
sae1 = joblib.load(MODELS / "SAE" / "sae_model_demo.joblib")
for _, slide in slides.items():
    ms.tl.feature_extraction(
        slide,
        model="uni",
        tile_key="tiles",
        key_added="UNI_embedding",
        sparse=True,
        sparse_transform=sae1.transform,
        sparse_key_added="UNI_SAE_demo",
        save=False,
    )
selector1 = FeatureSelector(pct_threshold=0.001, max_score_threshold=0.01, n_chunks=40)
selector1.compute_activation_stats(slides, feature_prefix='UNI_SAE_demo', num_features=65536)
sae1_idx = selector1.get_selected_indices()


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

W1 = F.normalize(sae1.model_.decoder.weight.detach().cpu(), dim=0)
W2 = F.normalize(sae.model_.decoder.weight.detach().cpu(), dim=0)
best = (W1[:, sae1_idx].T @ W2).max(dim=1).values  # best match in new SAE for each SAE1 feature

print(f"Selected features: SAE1 {len(sae1_idx)}, per-sample {len(highly_active_features_idx)}")
print(f"Shared selected indices: {len(np.intersect1d(sae1_idx, highly_active_features_idx))}")
print(f"SAE1 selected features with a decoder match (cosine > 0.9): {(best > 0.9).float().mean():.1%}")
